In [2]:
import os

def polacz_kod_zrodlowy(prefix_wyjsciowy="kod_zrodlowy"):
    limit_linii_na_plik_wyjsciowy = 5000
    limit_linii_na_pojedynczy_plik = 1000 # Zaostrzamy! Odcinamy wszystko co ma ponad 1000 linii
    
    ignorowane_foldery = {
        'node_modules', '__pycache__', 'dist', 'build', 
        'venv', 'env', 'coverage', 'target', 'out', '.git', '.next', '.idea', 'deprecated',
        # 'backend',
        # 'frontend',
    }
    
    # TWARDA CZARNA LISTA - wpisujemy tu Twojego potwora
    ignorowane_pliki = {
        'package.json', 'package-lock.json', 'yarn.lock', 'pnpm-lock.yaml', 
        'poetry.lock', 'Gemfile.lock', 'bun.lockb', 'tsconfig.json'
    }

    dozwolone_rozszerzenia = {
        '.py', '.ts', '.tsx', '.js', '.jsx', 
        '.html', '.css', '.json', '.txt',
        '.java', '.cpp', '.c', '.h', '.cs', '.md'
    }

    aktualny_plik_nr = 1
    aktualna_liczba_linii = 0

    def otworz_nowy_plik(nr):
        return open(f"{prefix_wyjsciowy}_czesc_{nr}.txt", 'w', encoding='utf-8')

    outfile = otworz_nowy_plik(aktualny_plik_nr)

    try:
        for root, dirs, files in os.walk('.'):
            # Filtrujemy foldery
            dirs[:] = [d for d in dirs if d not in ignorowane_foldery and not d.startswith('.')]

            for file in files:
                # Odrzucamy pliki z czarnej listy (w tym package.json)
                if file in ignorowane_pliki:
                    continue

                if any(file.endswith(ext) for ext in dozwolone_rozszerzenia):
                    filepath = os.path.join(root, file)
                    
                    try:
                        with open(filepath, 'r', encoding='utf-8') as infile:
                            linie_pliku = infile.readlines()
                            liczba_linii_w_pliku = len(linie_pliku)

                            # Pomijamy puste pliki
                            if liczba_linii_w_pliku == 0:
                                continue

                            # ZABEZPIECZENIE: Odrzucamy mutanty
                            if liczba_linii_w_pliku > limit_linii_na_pojedynczy_plik:
                                print(f"[ZIGNOROWANO] Plik {filepath} to potwór ({liczba_linii_w_pliku} linii).")
                                continue

                            naglowek = f"\n\n{'='*60}\n--- PLIK: {filepath} ---\n{'='*60}\n\n"
                            naglowek_linie = naglowek.count('\n')
                            calkowita_liczba_dodawanych_linii = liczba_linii_w_pliku + naglowek_linie

                            # Dzielenie na pliki po maks 5000 linii
                            if aktualna_liczba_linii + calkowita_liczba_dodawanych_linii > limit_linii_na_plik_wyjsciowy and aktualna_liczba_linii > 0:
                                outfile.close()
                                aktualny_plik_nr += 1
                                outfile = otworz_nowy_plik(aktualny_plik_nr)
                                aktualna_liczba_linii = 0
                                print(f"-> Utworzono kolejną paczkę: {prefix_wyjsciowy}_czesc_{aktualny_plik_nr}.txt")

                            # Zapis
                            outfile.write(naglowek)
                            outfile.writelines(linie_pliku)
                            aktualna_liczba_linii += calkowita_liczba_dodawanych_linii

                    except Exception as e:
                        print(f"[Błąd odczytu] {filepath}: {e}")

    finally:
        outfile.close()
        
    print(f"\nSukces! Kod czysty jak łza, podzielony na {aktualny_plik_nr} plików.")

if __name__ == "__main__":
    polacz_kod_zrodlowy()

[ZIGNOROWANO] Plik .\kod_zrodlowy_czesc_2.txt to potwór (4050 linii).
-> Utworzono kolejną paczkę: kod_zrodlowy_czesc_2.txt

Sukces! Kod czysty jak łza, podzielony na 2 plików.
